In [ ]:
import re
import sys
import json
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from us import states
from census import Census
 
import useful
from setup import CENSUS_API_KEY 

target_years = [2024]

In [17]:
def get_landscape_data(target_years, loud=False):

    prefixes = {
        "B01003_001": "tot_pop",

        "B25003_001": "tot_occ",
        "B25003_003": "rent_occ",

        "B25069_002": "pay_util",
        "B25069_003": "not_pay_util",

        # income
        "B25119_001": "med_in",
        "B25119_002": "med_in_own",
        "B25119_003": "med_in_rnt",
        
        "B25070_002": "perc_rent_0-10",
        "B25070_003": "perc_rent_10-14.9",
        "B25070_004": "perc_rent_15-19.9",
        "B25070_005": "perc_rent_20-24.9",
        "B25070_006": "perc_rent_25-29.9",
        "B25070_007": "perc_rent_30-34.9",
        "B25070_008": "perc_rent_35-39.9",
        "B25070_009": "perc_rent_40-44.9",
        "B25070_010": "perc_rent_50-100",
        "B25070_011": "perc_rent_null"
    }
    # landscape_variables = vars_and_moes(prefixes)
    
    output = useful.cousub_states_years_variables(
        useful.target_states, target_years, prefixes, 
        loud=loud)
    
    output = useful.aggregate(
        df=output, 
        agg_name="rent_gt_30_count",
        to_agg=[
            "perc_rent_30-34.9",
            "perc_rent_35-39.9",
            "perc_rent_40-44.9",
            "perc_rent_50-100"],
        drop=False)
    
    output = useful.aggregate(
        df=output, 
        agg_name="rent_lt_30_count",
        to_agg=[
            "perc_rent_0-10",
            "perc_rent_10-14.9",
            "perc_rent_15-19.9",
            "perc_rent_20-24.9",
            "perc_rent_25-29.9"],
        drop=False)
    
    output = useful.proportion(df=output, num="rent_gt_30_count", denom="rent_occ", new_name="rent_gt_30")
    output = useful.proportion(df=output, num="rent_lt_30_count", denom="rent_occ", new_name="rent_lt_30")
    output = useful.proportion(df=output, num="rent_occ", denom="tot_occ", new_name="rent_sh")

    output.sort_values(by=["GEOID", "year"], inplace=True)
    first_cols = [
        "year", "GEOID", "tot_occ", "rent_occ", "rent_sh", 
        "rent_gt_30", "rent_lt_30"]
    last_cols = [col for col in output.columns if col not in first_cols]
    output = output[first_cols + last_cols]

    return output

In [21]:
# Source: https://www.census.gov/cgi-bin/geo/shapefiles/index.php

cousub_gdf =  gpd.GeoDataFrame(
    pd.concat([
        gpd.read_file('../shapes/tl_2024_09_cousub'),
        gpd.read_file('../shapes/tl_2024_23_cousub'), 
        gpd.read_file('../shapes/tl_2024_25_cousub'),
        gpd.read_file('../shapes/tl_2024_33_cousub'),
        gpd.read_file('../shapes/tl_2024_44_cousub'),
        gpd.read_file('../shapes/tl_2024_50_cousub'),
    ])
).merge(
    get_landscape_data(
    target_years=target_years,
    loud=False), 
    on="GEOID", 
    how="inner"
).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.axis('off')

ct_state = gpd.read_file(states.CT.shapefile_urls()['state'])
me_state = gpd.read_file(states.ME.shapefile_urls()['state'])
ma_state = gpd.read_file(states.MA.shapefile_urls()['state'])
nh_state = gpd.read_file(states.NH.shapefile_urls()['state'])
ri_state = gpd.read_file(states.RI.shapefile_urls()['state'])
vt_state = gpd.read_file(states.VT.shapefile_urls()['state'])

ct_state.plot(ax=ax, facecolor='none')
me_state.plot(ax=ax, facecolor='none')
ma_state.plot(ax=ax, facecolor='none')
nh_state.plot(ax=ax, facecolor='none')
ri_state.plot(ax=ax, facecolor='none')
vt_state.plot(ax=ax, facecolor='none')

cousub_gdf.loc[cousub_gdf['nems']==True].plot(ax=ax, column='rent_sh', cmap='Purples')
cousub_gdf.loc[cousub_gdf['nems']==False].plot(ax=ax, column='rent_sh', cmap='Greens')

In [28]:
print('Some interesting numbers.')
print('Percent high rent burden in NE: ', round( 100.00*cousub_gdf['rent_gt_30_count'].sum()/cousub_gdf['rent_occ'].sum(), 2))

print('Total population in NEMS Network: ',cousub_gdf.loc[cousub_gdf['nems']==True]['tot_pop'].sum())

Some interesting numbers.
Percent high rent burden in NE:  48.0
Total population in NEMS Network:  2663545.0
